# Day 5.2 — Model Configuration and Runtime
Configuration is data; execution is code. If every agent builds its own request, every agent has its
own bugs and its own way of hiding a cost. One **adapter** hides the provider; the event log records
what each call cost. The loop that drives them is assembled in 5.4, once a registry and a policy exist.

### How the pieces divide

A provider adapter hides API-specific shapes behind one method,
`decide(prompt, config, tools, history) -> ModelDecision`. Swapping mock for OpenRouter must not
touch the registry, the policy or the loop.

### Step 1 — The mock provider reads the configuration, not the agent's name

`mock_plan` in the JSON file says which tools this agent should request — which is why a brand-new
configuration works in mock mode with no new Python.

In [ ]:
# Values used when a mock plan does not spell an argument out itself.
_PLACEHOLDER_BY_TYPE = {"string": "", "integer": 1, "number": 1, "boolean": True, "array": [], "object": {}}

def _fill(value, prompt):
    """Replace the literal token {prompt} inside strings coming from a mock plan."""
    if isinstance(value, str):
        return value.replace("{prompt}", prompt)
    if isinstance(value, dict):
        return {key: _fill(item, prompt) for key, item in value.items()}
    if isinstance(value, list):
        return [_fill(item, prompt) for item in value]
    return value

def default_arguments(spec, prompt):
    """Schema-valid arguments for a tool nobody wrote a plan for: required strings get the prompt."""
    arguments = {}
    for key in spec.input_schema.get("required", []):
        kind = spec.input_schema.get("properties", {}).get(key, {}).get("type", "string")
        arguments[key] = prompt if kind == "string" else _PLACEHOLDER_BY_TYPE.get(kind, prompt)
    return arguments

def applicable_plan(config, tools, prompt):
    """Turn config.mock_plan into the concrete steps this prompt should take."""
    visible = {spec.name: spec for spec in tools}
    steps = []
    for entry in config.mock_plan:
        if entry.get("tool") not in visible:
            continue                                   # not on this agent's allow-list -> cannot apply
        trigger = entry.get("when")
        if trigger and trigger.lower() not in prompt.lower():
            continue                                   # this step only applies to matching prompts
        arguments = (default_arguments(visible[entry["tool"]], prompt) if entry.get("arguments") is None
                     else _fill(entry["arguments"], prompt))
        steps.append({"tool": entry["tool"], "arguments": arguments})
    if steps or not tools:
        return steps
    first = tools[0]                                   # no plan at all: request the first visible tool once
    return [{"tool": first.name, "arguments": default_arguments(first, prompt)}]

class MockProvider:
    """Deterministic stand-in. It never pretends to be intelligent; it follows the config."""
    def decide(self, prompt, config, tools, history):
        steps = applicable_plan(config, tools, prompt)
        done = [item for item in history if item.get("role") == "tool"]   # results seen so far
        if len(done) < len(steps):
            step = steps[len(done)]
            return ModelDecision("tool", tool=step["tool"], arguments=step["arguments"])
        if done:
            return ModelDecision("final", content=f"Completed with tool result: {done[-1]['content']}")
        return ModelDecision("final", content="No permitted tool is needed.")

# Two tool descriptions by hand (5.3 builds the registry that normally supplies them).
NOTES_SPEC = ToolSpec("lookup_notes", "Look up a course note",
                      {"type": "object", "properties": {"query": {"type": "string"}},
                       "required": ["query"], "additionalProperties": False}, "read")
EMAIL_SPEC = ToolSpec("send_email", "Send a simulated external email",
                      {"type": "object", "properties": {"to": {"type": "string"}, "subject": {"type": "string"},
                                                        "body": {"type": "string"}},
                       "required": ["to", "subject", "body"], "additionalProperties": False}, "external")

task = load_config("task_agent")
print("task_agent plan for a prompt containing 'send':")
for step in applicable_plan(task, [NOTES_SPEC, EMAIL_SPEC], "Please send the update"):
    print("   ", step)
print("task_agent plan for a prompt that does not:")
for step in applicable_plan(task, [NOTES_SPEC, EMAIL_SPEC], "Prepare the update"):
    print("   ", step)
print()
print("First decision returned to the runtime:")
print("  ", MockProvider().decide("Please send the update", task, [NOTES_SPEC, EMAIL_SPEC], []))

### Step 2 — The live adapter wraps Day 1's `chat()`

Same method, same return type: the runtime cannot tell the two apart.

In [ ]:
class LiveProvider:
    """OpenRouter adapter. It reuses chat() from the setup cell and returns a ModelDecision."""
    def __init__(self, config):
        self.config = config

    def decide(self, prompt, config, tools, history):
        messages = [{"role": "system", "content": config.instructions}, {"role": "user", "content": prompt}]
        for item in history:                                    # replay what already happened
            messages.append({"role": "user", "content": f"[{item.get('name', 'step')}] {item.get('content', '')}"}
                            if item.get("role") == "tool" else
                            {"role": "assistant", "content": f"requested {item.get('tool')} {item.get('arguments')}"})
        schemas = [{"type": "function", "function": {"name": t.name, "description": t.description,
                                                     "parameters": t.input_schema}} for t in tools]
        reply = chat(messages, tools=schemas or None, temperature=self.config.temperature,
                     max_tokens=self.config.max_output_tokens)
        usage = {"prompt_tokens": reply["usage"]["prompt_tokens"],
                 "completion_tokens": reply["usage"]["completion_tokens"], "cost_usd": 0.0}
        if reply["tool_calls"]:
            call = reply["tool_calls"][0]
            return ModelDecision("tool", tool=call["name"], arguments=call["arguments"], usage=usage)
        return ModelDecision("final", content=reply["content"], usage=usage)

def build_provider(model_config):
    """The ONE place that knows how each provider is reached."""
    if model_config.provider == "mock":
        return MockProvider()
    if model_config.provider == "openrouter":
        return LiveProvider(model_config)
    raise ValueError(f"Unsupported provider: {model_config.provider!r}")

research = load_config("research_agent")
if LIVE:
    research.model.provider = "openrouter"       # a key is present: point the SAME config at the real model
    research.model.model = MODEL

provider = build_provider(research.model)
print("Configured provider:", research.model.provider, "->", type(provider).__name__)
print("Model name         :", research.model.model)
print("The runtime is handed this object and never asks what it is.")

### Step 3 — Run one decision, and fall back if the network misbehaves

Every live call is wrapped. One timeout must not stop the class: report it in one line, continue on
the mock provider.

In [ ]:
try:
    decision = provider.decide("harness", research, [NOTES_SPEC], [])
except Exception as exc:                       # any provider failure at all
    print("Live provider failed:", type(exc).__name__, exc)
    print("Falling back to the mock provider so the lesson continues.")
    research.model.provider = "mock"
    provider = MockProvider()
    decision = provider.decide("harness", research, [NOTES_SPEC], [])

print("Decision kind :", decision.kind)
print("Tool requested:", decision.tool, decision.arguments)
print("Content       :", decision.content[:120] or "(none - it asked for a tool instead)")
print("Usage         :", decision.usage or "{} - mock mode buys nothing, so nothing is counted")

### Step 4 — Events: where a run's story (and its bill) is kept

An **event** is an append-only observation. Usage rides on *every* model call, so cost is
attributable to a whole run — failed attempts included.

In [ ]:
from datetime import datetime, timezone

class EventStore:
    """Append-only observations, in memory and (from 5.5) on disk as JSON lines."""
    def __init__(self, path=None):
        self.path = Path(path) if path else None
        self.by_run = {}

    def add(self, run_id, event, **details):
        row = {"run_id": run_id, "event": event,
               "timestamp": datetime.now(timezone.utc).isoformat(), "details": details}
        self.by_run.setdefault(run_id, []).append(row)
        if self.path:                                   # durable: one JSON object per line
            self.path.parent.mkdir(parents=True, exist_ok=True)
            with self.path.open("a", encoding="utf-8") as handle:
                handle.write(json.dumps(row) + "\n")
        return row

    def get(self, run_id):
        return list(self.by_run.get(run_id, []))

events = EventStore()
events.add("demo-run", "model_requested", step=1, visible_tools=["lookup_notes"])
events.add("demo-run", "model_completed", step=1, usage=decision.usage)

for row in events.get("demo-run"):
    print(f"{row['event']:<18}", row["details"])
print()
print("Same three fields whichever provider ran: run_id, event, details.")
print("That is what makes two runs comparable at all.")

### Step 5 — A provider call that survives a hiccup

Network calls fail. The runtime retries a **bounded** number of times with exponential backoff and
records every attempt; errors that cannot improve are not retried at all.

In [ ]:
import time

MAX_PROVIDER_RETRIES = 2            # extra attempts after the first one
RETRY_BASE_DELAY_SECONDS = 0.05     # tiny on purpose: a classroom demo must stay fast
# Bad arguments, missing keys and misconfiguration will not improve on repetition.
NON_RETRYABLE_ERRORS = (ValueError, TypeError, KeyError)

def call_provider_with_retries(provider, prompt, config, tools, history, events, run_id, step):
    """Call the provider; retry transient failures with exponential backoff, recording each attempt."""
    attempt = 0
    while True:
        attempt += 1
        try:
            return provider.decide(prompt, config, tools, history)
        except NON_RETRYABLE_ERRORS as exc:
            events.add(run_id, "provider_error_not_retried", step=step, error=f"{type(exc).__name__}: {exc}")
            raise
        except Exception as exc:
            if attempt > MAX_PROVIDER_RETRIES:
                events.add(run_id, "provider_retry_budget_exhausted", step=step, attempts=attempt, error=str(exc))
                raise
            delay = RETRY_BASE_DELAY_SECONDS * (2 ** (attempt - 1))     # 0.05s, then 0.10s
            events.add(run_id, "provider_retry", step=step, attempt=attempt, error=str(exc),
                       retry_in_seconds=round(delay, 3), retries_left=MAX_PROVIDER_RETRIES - attempt + 1)
            time.sleep(delay)

print("Retry budget      :", MAX_PROVIDER_RETRIES, "extra attempts per model call")
print("Backoff schedule  :", [round(RETRY_BASE_DELAY_SECONDS * 2 ** n, 3) for n in range(MAX_PROVIDER_RETRIES)], "seconds")
print("Never retried     :", [e.__name__ for e in NON_RETRYABLE_ERRORS])
print()
print("Proof that it is transparent when nothing goes wrong:")
print("  ", call_provider_with_retries(MockProvider(), "harness", research, [NOTES_SPEC], [], events, "demo-run", 1))

### Try it yourself

A configuration names a provider that does not exist. Does the harness guess, or stop?

In [ ]:
# --- Worked solution ---------------------------------------------------------------
# build_provider knows exactly two provider names. Anything else is a configuration
# error, raised BEFORE a request is built or a key is read.
typo = ModelConfig(provider="openrouetr", model=MODEL)          # deliberate typo
try:
    build_provider(typo)
except ValueError as exc:
    print("Rejected:", exc)
    print("-> good: a typo can never silently become 'no model at all'.")

print()
print("Where each setting belongs - the most common configuration bug in agent projects:")
print("  temperature, max_output_tokens -> ModelConfig  (how the model writes) ->", research.model)
print("  max_steps, allowed_tools       -> AgentConfig  (what the agent may do) ->",
      research.max_steps, research.allowed_tools)
print("  OPENROUTER_API_KEY             -> the session   (a secret; never JSON) -> present:", LIVE)

### Checkpoint

**1. Why does `build_provider` exist instead of the loop calling the API directly?**

<details><summary>Show answer</summary>

So one vendor's request format cannot leak into the loop. The runtime only knows `provider.decide(...) -> ModelDecision`, so swapping adapters is a configuration change.

</details>

**2. In mock mode the usage record is empty. Is that a bug?**

<details><summary>Show answer</summary>

No. Nothing was bought, so there is nothing to report — but the *field* exists in every model event, so the same reading code works on a live run.

</details>

### Recap

- **Limitation seen:** agents building their own requests duplicate bugs, timeouts and cost handling.
- **Layer added:** one `decide()` contract, two adapters chosen by data, an event store, a bounded retry.
- **Evidence:** one config, one `ModelDecision`, usage in an event, and a mistyped provider refused up front.